In [ ]:
import os
import json
import pandas as pd
# show all rows
pd.set_option('display.max_rows', None)
# show all columns
pd.set_option('display.max_columns', None)
# show whole value
pd.set_option('display.width', None)

import google_auth_oauthlib.flow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

### List out items on a youtube playlist, given
- client_secret.json that details the oauth credentials
- playlist id

In [34]:
def getYoutubeClient():
    # Disable OAuthlib's HTTPS verification when running locally.
    # *DO NOT* leave this option enabled in production.
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

    api_service_name = "youtube"
    api_version = "v3"
    client_secrets_file = "client_secret.json"
    scopes = ["https://www.googleapis.com/auth/youtube.readonly"]
    token_file = "token.json"

    credentials = None
    
    # checks for existing credentials from previous run
    if os.path.exists(token_file):
        credentials = Credentials.from_authorized_user_file(token_file, scopes)

    # if no valid credentials are available, get new credentials and create an API client
    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
        else: # else, get credentials and create an API client
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
                client_secrets_file, scopes)
            credentials = flow.run_local_server(port=0)

        # save credentials for next run
        with open(token_file, "w") as token:
            token.write(credentials.to_json())

    return googleapiclient.discovery.build(api_service_name, api_version, credentials=credentials)

def getSongNames(playlist_id):
    # gets credentials/tokenized info to be able to run YouTube API requests
    youtube = getYoutubeClient()

    try: 
        # lists out the items in the playlist, with a max of x results
        request = youtube.playlistItems().list(
            part="snippet,contentDetails",
            maxResults=5,
            playlistId=playlist_id
        )
        response = request.execute()

        # extract song/item name from the nested maps
        # 'items' key is list of maps -> each map has 'snippet' key -> has 'title' key, which is the song name as the value
        items = response.get("items", [])

        song_data = []

        # iterate through the items maps to extract each song
        for item in items:
            snippet = item.get("snippet", {})
            title = snippet.get("title", "")
            song_data.append(title)

        # return a DataFrame from the song data
        return pd.DataFrame(song_data, columns=["Song_Name"])
    except HttpError as e:
        print(f"An HTTP error {e.resp.status} occurred:\n{e.content}")
        return None

In [35]:
if __name__ == "__main__":
    # test
    myPlaylistId = "PL2vR6i7V0rIoXdJ7HZT67afj4T24Ji0Od"
    df = getSongNames(myPlaylistId)
    print(df)

                                           Song_Name
0         Foster The People - Sit Next to Me (Audio)
1                      Kim Tae Woo - 사랑비 (Love Rain)
2  The Pussycat Dolls - Don't Cha (Official Music...
3           Rain - Brothers & me (Eng Sub + Romanji)
4                      Years & Years - King (lyrics)
